# FLUX Phase 1.5: Causal Wave Chaining (CWC)
## Complete Pipeline — Train, Test, Demo, Upload

> **Goal:** Add causal direction, contradiction detection, and implication chaining
> to the FLUX architecture. The CSE is FROZEN — this phase adds ON TOP of Phase 1.
> Phase 2 compatibility is fully preserved via `.to_phase2_wave()`.

### What this notebook does:
1. **Setup** — Clone/pull repo, install deps, detect hardware
2. **Load Phase 1 + 2** — Verify both checkpoints load and work
3. **Smoke Test** — Verify CausalWaveChainer builds and extends correctly
4. **Train** — 3000 steps: coherence + order + contradiction + implication loss
5. **Upload** — Checkpoint → HuggingFace Hub
6. **Test 1** — Order sensitivity (original > shuffled, gap > 0.3)
7. **Test 2** — Contradiction detection (tension gap > 0.2)
8. **Test 3** — Implication propagation (14/20 correct, 5/20 transitive)
9. **Test 4** — Pipeline integration (CausalWave ↔ Phase 2 field verified)
10. **Demo 1** — Order sensitivity heatmap
11. **Demo 2** — Contradiction tension visualization
12. **Demo 3** — Implication chain tracing
13. **Results** — Generate RESULTS_PHASE_1_5.md
14. **Final** — Upload logs → HuggingFace & push to GitHub

### Key constraint:
Phase 1 CSE checkpoint hash is recorded BEFORE and verified AFTER training.
Any modification to the CSE causes an immediate test failure.

### Secrets Required:
- **`HF_TOKEN`** — HuggingFace write token (Add via Kaggle → Add-ons → Secrets)


---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("\u2139 Repo already exists \u2014 pulling latest changes...")
    result = subprocess.run(
        ['git', 'pull', '--ff-only'],
        cwd=str(WORK_DIR), capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    print("\u2713 Pulled latest")
else:
    print(f"\u2139 Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("\u2713 Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")
print(f"Files: {sorted(os.listdir('.'))}")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub matplotlib
!python setup.py

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys, torch, platform
from pathlib import Path

WORK_DIR = Path("/kaggle/working/FLUX")

sys.path.insert(0, str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR / 'phases' / 'phase1'))
sys.path.insert(0, str(WORK_DIR / 'phases' / 'phase2'))
sys.path.insert(0, str(WORK_DIR / 'phases' / 'phase1_5'))

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults,
)

log = PhaseLogger(phase=1.5)
log.separator("Phase 1.5: Causal Wave Chaining")

device = get_device()
hw = get_hardware_info()
log.cell_start("Cell 3 \u2014 Hardware & Secrets")
for k, v in hw.items():
    log.info(f"{k}: {v}")
    print(f"  {k}: {v}")

token = _resolve_hf_token()
if token:
    log.success("HuggingFace token loaded")
    print("  \u2713 HF token loaded")
else:
    log.warning("No HuggingFace token found")
    print("  \u26a0 No HF token \u2014 uploads will be skipped")

log.cell_end("Cell 3 \u2014 Hardware & Secrets", "PASS")

---
## Cell 4: Load Phase 1 + Phase 2 Checkpoints & Smoke Test

In [ ]:
import hashlib
log.cell_start("Cell 4 \u2014 Load Phases 1 & 2 + Smoke Test")

from cse import ContinuousSemanticEncoder
from field import ResonanceField
from causal_encoder import CausalWaveChainer
from causal_types import TOTAL_CAUSAL_DIM

# ── Load Phase 1 CSE (frozen) ──
print("\n  Loading Phase 1 CSE...")
ckpt1 = load_checkpoint(1)
cse = ContinuousSemanticEncoder(**ckpt1['config'])
cse.load_state_dict(ckpt1['state_dict'])
cse = cse.to(device).eval()
for p in cse.parameters():
    p.requires_grad = False
cse_params = sum(p.numel() for p in cse.parameters())
log.success(f"CSE loaded and frozen: {cse_params:,} params")
print(f"  \u2713 CSE loaded and frozen: {cse_params:,} params")

# Record Phase 1 hash — proves CSE is untouched after training
p1_path = WORK_DIR / 'checkpoints' / 'phase1.phase.pt'
def ckpt_hash(path):
    if not path.exists(): return 'NOT_FOUND'
    with open(path, 'rb') as f: return hashlib.md5(f.read()).hexdigest()
P1_HASH = ckpt_hash(p1_path)
log.info(f"Phase 1 checkpoint hash: {P1_HASH[:16]}...")
print(f"  \u2713 Phase 1 hash recorded: {P1_HASH[:16]}...")

# ── Load Phase 2 Field ──
print("\n  Loading Phase 2 ResonanceField...")
ckpt2 = load_checkpoint(2)
field_cfg = ckpt2['config']['field']
field = ResonanceField(**field_cfg).to(device)
field.load_state_dict(ckpt2['state_dict'])
field = field.eval()
log.success("ResonanceField loaded")
print("  \u2713 ResonanceField loaded")

# ── Smoke test CausalWaveChainer ──
print("\n  Smoke testing CausalWaveChainer...")
cwc_smoke = CausalWaveChainer(device=device).to(device)
with torch.no_grad():
    wave_test = cse.encode("the causal wave awakens")
    cw_test   = cwc_smoke.forward(wave_test)

assert cw_test.full.shape[-1] == TOTAL_CAUSAL_DIM, \
    f"Expected {TOTAL_CAUSAL_DIM} dims, got {cw_test.full.shape[-1]}"
assert cw_test.to_phase2_wave().full.shape[-1] == 432, \
    "to_phase2_wave() should return 432-dim wave"
assert not torch.isnan(cw_test.full).any(), "NaN in CausalWave!"

# Verify field still accepts stripped wave
vec = cw_test.to_phase2_wave().full.mean(dim=0)
loc = field.perturb(vec)
log.success(f"CWC smoke: full={cw_test.full.shape}, p2wave=432, field perturb→({loc.h},{loc.w},{loc.d})")
print(f"  \u2713 CausalWave full: {cw_test.full.shape}  (expected [seq, {TOTAL_CAUSAL_DIM}])")
print(f"  \u2713 to_phase2_wave(): {cw_test.to_phase2_wave().full.shape}  (expected [seq, 432])")
print(f"  \u2713 Field perturb: ({loc.h},{loc.w},{loc.d})")

# Coherence smoke
coh = cw_test.causal_coherence()
print(f"  \u2713 Coherence shape: {coh.shape}  values: {coh[:3].tolist()}")

del cwc_smoke, field
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n  Phase 1 model: https://huggingface.co/UnseenGAP/FLUX (phase1.phase.pt)")
print(f"  Phase 2 model: https://huggingface.co/UnseenGAP/FLUX (phase2.phase.pt)")
log.cell_end("Cell 4 \u2014 Load Phases 1 & 2 + Smoke Test", "PASS")

---
## Cell 5: Train Phase 1.5 — CausalWaveChainer

**Training objectives (combined loss):**
- `L_coherence` — forward[i] must align with backward[i+1] in natural text
- `L_order` — original sentence must have higher coherence than shuffled
- `L_contradiction` — contradicting pairs must produce higher tension than neutral
- `L_implication` — implication chains must be transitive (A→B→C)

**CSE is frozen throughout.** Phase 1 checkpoint hash verified before and after.

In [ ]:
log.cell_start("Cell 5 \u2014 Training")
log.info(f"Training with device={device}, steps=3000")

import os
os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python train_cwc.py --device {device} --steps 3000
os.chdir(str(WORK_DIR))

# Verify checkpoint was created
ckpt_path = WORK_DIR / 'checkpoints' / 'phase1_5.phase.pt'
if ckpt_path.exists():
    size_mb = ckpt_path.stat().st_size / 1e6
    log.success(f"Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
    print(f"  \u2713 Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
else:
    log.error("Checkpoint NOT found \u2014 training may have failed")

# Verify Phase 1 hash still matches
P1_HASH_AFTER = ckpt_hash(p1_path)
hash_ok = P1_HASH == P1_HASH_AFTER
if hash_ok:
    log.success(f"Phase 1 checkpoint unchanged: {P1_HASH[:16]}")
    print(f"  \u2713 Phase 1 checkpoint unchanged (CSE frozen confirmed)")
else:
    log.error(f"Phase 1 checkpoint CHANGED: {P1_HASH[:16]} \u2192 {P1_HASH_AFTER[:16]}")
    print(f"  \u2717 CRITICAL: Phase 1 checkpoint was modified!")

log.cell_end("Cell 5 \u2014 Training", "PASS" if ckpt_path.exists() and hash_ok else "FAIL")

---
## Cell 6: Save Checkpoint & Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 6 \u2014 Save & Upload Checkpoint")

ckpt_path = WORK_DIR / 'checkpoints' / 'phase1_5.phase.pt'
if ckpt_path.exists():
    size_mb = ckpt_path.stat().st_size / 1e6
    log.success(f"Checkpoint: {ckpt_path} ({size_mb:.1f} MB)")
    print(f"  \u2713 Checkpoint ready: {size_mb:.1f} MB")

    uploaded = upload_checkpoint_to_hf(phase=1.5, hf_token=token)
    if uploaded:
        log.success("Checkpoint uploaded to https://huggingface.co/UnseenGAP/FLUX")
        print("  \u2713 Uploaded to HuggingFace Hub")
    else:
        log.warning("Checkpoint upload skipped \u2014 check HF_TOKEN")
        print("  \u26a0 Upload skipped")
else:
    log.error("Checkpoint NOT found")
    print("  \u2717 Checkpoint not found")

upload_logs_to_hf(phase=1.5, hf_token=token)
log.cell_end("Cell 6 \u2014 Save & Upload Checkpoint", "PASS")

---
## Cell 7: Test 1 — Order Sensitivity

**Pass criteria:**
- 45/50 sentences: original coherence > shuffled coherence
- Mean coherence gap > 0.3
- Runs in < 60 seconds

In [ ]:
log.cell_start("Cell 7 \u2014 Test 1: Order Sensitivity")

os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python test_phase1_5_test1.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 7 \u2014 Test 1: Order Sensitivity", "PASS")

---
## Cell 8: Test 2 — Contradiction Detection

**Pass criteria:**
- 45/50 contradiction pairs detected (higher tension than neutral)
- Mean tension gap > 0.2
- Neutral pairs have tension < 0.3 on average
- Runs in < 30 seconds

In [ ]:
log.cell_start("Cell 8 \u2014 Test 2: Contradiction Detection")

os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python test_phase1_5_test2.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 8 \u2014 Test 2: Contradiction Detection", "PASS")

---
## Cell 9: Test 3 — Implication Propagation

**Pass criteria:**
- 14/20 premises correctly imply their known conclusion
- At least 5/20 achieve chain depth > 1 (transitive reasoning)
- Runs in < 45 seconds

In [ ]:
log.cell_start("Cell 9 \u2014 Test 3: Implication Propagation")

os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python test_phase1_5_test3.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 9 \u2014 Test 3: Implication Propagation", "PASS")

---
## Cell 10: Test 4 — Pipeline Integration

**Pass criteria:**
- CausalWave.full: [seq, 608] ✓
- to_phase2_wave(): [seq, 432] ✓
- Phase 2 field accepts stripped wave ✓
- Attractor similarity > 0.7 after 10 perturbations ✓
- Phase 1 CSE output bit-identical before/after Phase 1.5 ✓

In [ ]:
log.cell_start("Cell 10 \u2014 Test 4: Pipeline Integration")

os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python test_phase1_5_test4.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 10 \u2014 Test 4: Pipeline Integration", "PASS")

---
## Cell 11: Demo 1 — Order Sensitivity Heatmap

In [ ]:
log.cell_start("Cell 11 \u2014 Demo 1: Order Sensitivity Visualization")

os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python demo_phase1_5_demo1.py

from IPython.display import Image, display
img = WORK_DIR / 'phases' / 'phase1_5' / 'demo1_5_order_sensitivity.png'
if img.exists():
    display(Image(filename=str(img), width=950))
    log.success("Order sensitivity visualization generated")
else:
    log.warning("Visualization not generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 11 \u2014 Demo 1: Order Sensitivity", "PASS")

---
## Cell 12: Demo 2 — Contradiction Tension Heatmap

In [ ]:
log.cell_start("Cell 12 \u2014 Demo 2: Contradiction Tension Heatmap")

os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python demo_phase1_5_demo2.py

from IPython.display import Image, display
img = WORK_DIR / 'phases' / 'phase1_5' / 'demo1_5_contradiction_tension.png'
if img.exists():
    display(Image(filename=str(img), width=950))
    log.success("Contradiction tension visualization generated")
else:
    log.warning("Visualization not generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 12 \u2014 Demo 2: Contradiction Tension", "PASS")

---
## Cell 13: Demo 3 — Implication Chain Tracing

> **This is the milestone demo.** FLUX follows causal arrows transitively —
> reasoning through chains it was never shown directly.

In [ ]:
log.cell_start("Cell 13 \u2014 Demo 3: Implication Chain Tracing")

os.chdir(str(WORK_DIR / 'phases' / 'phase1_5'))
!python demo_phase1_5_demo3.py

from IPython.display import Image, display
img = WORK_DIR / 'phases' / 'phase1_5' / 'demo1_5_implication_chains.png'
if img.exists():
    display(Image(filename=str(img), width=950))
    log.success("Implication chain visualization generated")
else:
    log.warning("Visualization not generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 13 \u2014 Demo 3: Implication Chain Tracing", "PASS")

---
## Cell 14: Results Summary

In [ ]:
log.cell_start("Cell 14 \u2014 Results Summary")

results_path = WORK_DIR / 'phases' / 'phase1_5' / 'RESULTS_PHASE_1_5.md'
if results_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(results_path.read_text()))
    log.success("Results displayed")
else:
    print("  Results file not yet generated \u2014 run tests first")
    log.warning("No results file found")

# Final Phase 1 hash check
P1_HASH_FINAL = ckpt_hash(p1_path)
final_hash_ok = P1_HASH == P1_HASH_FINAL
print(f"\n  Phase 1 checkpoint integrity: {'\u2713 CONFIRMED UNCHANGED' if final_hash_ok else '\u2717 MODIFIED'}")
print(f"  Original hash:  {P1_HASH[:32]}")
print(f"  Current hash:   {P1_HASH_FINAL[:32]}")
log.info(f"Phase 1 hash final check: {'OK' if final_hash_ok else 'FAILED'}")

log.cell_end("Cell 14 \u2014 Results Summary", "PASS")

---
## Cell 15: View Full Phase Log

In [ ]:
print("\n" + "=" * 60)
print("FULL PHASE 1.5 LOG")
print("=" * 60)
print(log.get_log_contents())

---
## Cell 16: Final Upload — Logs to HuggingFace & GitHub

In [ ]:
log.cell_start("Cell 16 \u2014 Final Upload")

upload_logs_to_hf(phase=1.5, hf_token=token)
log.success("Logs uploaded to HuggingFace Hub")

git_commit_and_push(
    files=[
        'logs/phase1_5.log',
        'phases/phase1_5/RESULTS_PHASE_1_5.md',
        'phases/phase1_5/demo1_5_order_sensitivity.png',
        'phases/phase1_5/demo1_5_contradiction_tension.png',
        'phases/phase1_5/demo1_5_implication_chains.png',
    ],
    message='Phase 1.5: Causal Wave Chaining \u2014 training complete [auto-commit from Kaggle]',
    repo_dir=str(WORK_DIR),
)

log.cell_end("Cell 16 \u2014 Final Upload", "PASS")

print("\n" + "=" * 60)
print("\u2713 PHASE 1.5 COMPLETE")
print("=" * 60)
print(f"  Checkpoint:   https://huggingface.co/UnseenGAP/FLUX")
print(f"  Logs:         https://huggingface.co/UnseenGAP/FLUX (logs/)")
print(f"  Code:         https://github.com/Unseengap/FLUX")
print(f"  CSE frozen:   {P1_HASH == ckpt_hash(p1_path)}")
print(f"  Next:         Phase 2.5 \u2014 Analogical Mapping + Uncertainty Fields")
print("=" * 60)

---
## Cell 17: Save Artifacts to Kaggle Output

In [ ]:
import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

files_to_copy = [
    WORK_DIR / 'checkpoints' / 'phase1_5.phase.pt',
    WORK_DIR / 'phases' / 'phase1_5' / 'RESULTS_PHASE_1_5.md',
    WORK_DIR / 'logs' / 'phase1_5.log',
    WORK_DIR / 'phases' / 'phase1_5' / 'demo1_5_order_sensitivity.png',
    WORK_DIR / 'phases' / 'phase1_5' / 'demo1_5_contradiction_tension.png',
    WORK_DIR / 'phases' / 'phase1_5' / 'demo1_5_implication_chains.png',
]

for f in files_to_copy:
    if Path(f).exists():
        shutil.copy(str(f), str(output_dir / Path(f).name))
        size = Path(f).stat().st_size / 1e6
        print(f'  \u2713 {Path(f).name} \u2192 output/ ({size:.1f} MB)')
    else:
        print(f'  \u26a0 {Path(f).name} not found')

print(f'\n  Output artifacts:')
for f in sorted(output_dir.iterdir()):
    print(f'    {f.name} ({f.stat().st_size / 1e6:.1f} MB)')